In [5]:
import pandas as pd
from pathlib import Path
import sys

In [6]:

ROOT_DIR = Path.cwd().parent

sys.path.append(str(ROOT_DIR))

from src.utils.paths import OLIST_DIR

## Check for customer dataset

In [9]:
customers = pd.read_csv(
    OLIST_DIR / "olist_customers_dataset.csv"
)

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [14]:
print(customers['customer_id'].nunique(), customers['customer_unique_id'].nunique())
print(customers.info())

99441 96096
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB
None


### Um id para cada compra, porém cada comprador tem seu unique ID. customer_unique_id serve de chave primária. zip_code_prefix é chave estrangeira da base geolocation.

# Check for Geoloc datasec

In [20]:
geoloc = pd.read_csv(
    OLIST_DIR / "olist_geolocation_dataset.csv"
)

geoloc.head()


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [19]:
geoloc.info()
print(geoloc['geolocation_zip_code_prefix'].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 50.1 MB
19015


In [26]:
geoloc[geoloc.duplicated(keep=False)].sort_values(by='geolocation_zip_code_prefix')

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
851,1001,-23.549825,-46.633970,sao paulo,SP
1062,1001,-23.550498,-46.634338,sao paulo,SP
1246,1001,-23.549292,-46.633559,sao paulo,SP
897,1001,-23.549292,-46.633559,sao paulo,SP
99,1001,-23.549292,-46.633559,sao paulo,SP
...,...,...,...,...,...
999775,99980,-28.386689,-51.847091,david canabarro,RS
999835,99980,-28.387432,-51.847727,david canabarro,RS
999958,99980,-28.387059,-51.848964,david canabarro,RS
1000133,99980,-28.386689,-51.847091,david canabarro,RS


### Tabela geolocation pode ser reduzida ao remover zip prefix duplicados, já que representam essencialmente a mesma geolocalizacao

# Check for order items dataset

In [27]:
order_items = pd.read_csv(
    OLIST_DIR / "olist_order_items_dataset.csv"
)

order_items.head()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [33]:
order_items['order_item_id'].value_counts()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

### order_item_id contém a quantidade de itens no mesmo pedido. O mesmo pedido pode conter itens diferentes. Exemplo pertinente abaixo, onde 20 pedidos sao para o mesmo item, mas o 21o é um diferente:


In [34]:
order_items[order_items['order_id'] == '8272b63d03f5f79c56e9e4120aec44ef']

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


### Esse dataset nao contém uma chave primária clara. Seria necessário criar uma chave composta (funciona como uma tabela fato dos pedidos)

# Check order_payments dataset

In [16]:
order_paymnt = pd.read_csv(
    OLIST_DIR / "olist_order_payments_dataset.csv"
)

order_paymnt.head()
order_paymnt.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB


In [9]:
order_paymnt['payment_sequential'].value_counts()

payment_sequential
1     99360
2      3039
3       581
4       278
5       170
6       118
7        82
8        54
9        43
10       34
11       29
12       21
13       13
14       10
15        8
17        6
19        6
16        6
18        6
21        4
20        4
22        3
25        2
26        2
24        2
23        2
27        1
29        1
28        1
Name: count, dtype: int64

In [11]:
order_paymnt[order_paymnt['payment_sequential'] == 10].sort_values(by='order_id')

,order_id,payment_sequential,payment_type,payment_installments,payment_value
42371,0bbb3f7791a87d0307555e57da3a1ff1,10,voucher,1,7.58
41710,1a611328643ae11146ba09a4425d2e12,10,voucher,1,6.34
83990,1be51feefcd481bee3118900e6777057,10,voucher,1,6.67
22352,1d9a9731b9c10fc9cba74e6f74782e8b,10,voucher,1,3.00
86473,1e6f350c900ca357945126e20117d293,10,voucher,1,17.59
4439,21577126c19bf11a0b91592e5844ba78,10,voucher,1,9.34
11663,27a940efdd448db29463b53ea0cfa2f4,10,voucher,1,6.06
88775,285c2e15bebd4ac83635ccc563dc71f4,10,voucher,1,2.78
18778,364f451ee38a4268d7c15d317021eb35,10,voucher,1,1.58
48985,370e2e6c1a9fd451eb7f0852daa3b006,10,voucher,1,13.79


In [15]:
# Check one order with 10 payment_sequential

order_paymnt.query('order_id == "0bbb3f7791a87d0307555e57da3a1ff1"')

,order_id,payment_sequential,payment_type,payment_installments,payment_value
2685,0bbb3f7791a87d0307555e57da3a1ff1,7,voucher,1,14.92
23387,0bbb3f7791a87d0307555e57da3a1ff1,2,voucher,1,2.41
31570,0bbb3f7791a87d0307555e57da3a1ff1,3,voucher,1,2.39
39298,0bbb3f7791a87d0307555e57da3a1ff1,11,voucher,1,21.68
42371,0bbb3f7791a87d0307555e57da3a1ff1,10,voucher,1,7.58
49652,0bbb3f7791a87d0307555e57da3a1ff1,4,voucher,1,2.58
53334,0bbb3f7791a87d0307555e57da3a1ff1,6,voucher,1,1.55
71125,0bbb3f7791a87d0307555e57da3a1ff1,5,voucher,1,2.58
73354,0bbb3f7791a87d0307555e57da3a1ff1,8,voucher,1,1.88
92290,0bbb3f7791a87d0307555e57da3a1ff1,1,credit_card,7,76.20


### Order_id repete de acordo com a quantidade de formas de pagamentos. Certos pedidos foram pagos com diversos meio de pagamentos; há uma linha para um deses meios. Chave primaria pode ser composta por id, payment sequential e payment_type

# Check order review

In [17]:
order_reviews = pd.read_csv(
    OLIST_DIR / "olist_order_reviews_dataset.csv"
)

order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
order_reviews['review_id'].nunique()